# Notebook 02 - Delta Lake, Time Travel e Schema Evolution

Neste notebook será realizada a criação da tabela Delta Lake a partir da camada Silver.

Também serão demonstradas as funcionalidades de:

- Delta Lake
- Time Travel
- Schema Evolution


In [1]:
import os
import shutil

import pandas as pd

from deltalake import DeltaTable
from deltalake.writer import write_deltalake

## Leitura da camada Silver


In [2]:
silver = pd.read_parquet(
    "../datalake/silver/silver.parquet"
)

print(f"Quantidade de registros: {len(silver)}")

silver.head()

Quantidade de registros: 23475


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,uop,_source_file,_batch_id,_ingested_at,nova_coluna,ano,mes,dia,total_vitimas,acidente_grave
0,753147,2026-02-18,QUARTA-FEIRA,13:40:00,RS,158,"565,9",SANTANA DO LIVRAMENTO,DESRESPEITAR A PREFERÊNCIA NO CRUZAMENTO,COLISÃO TRANSVERSAL,...,UOP01-DEL11-RS,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,2,18,2,False
1,758570,2026-03-15,DOMINGO,12:30:00,MG,40,701,BARBACENA,AUSÊNCIA DE REAÇÃO DO CONDUTOR,COLISÃO LATERAL MESMO SENTIDO,...,UOP02-DEL05-MG,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,3,15,1,False
2,747911,2026-01-24,SÁBADO,16:50:00,MG,381,"438,3",SANTA LUZIA,PISTA ESBURACADA,SAÍDA DE LEITO CARROÇÁVEL,...,UOP01-DEL01-MG,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,1,24,2,False
3,751057,2026-02-08,DOMINGO,20:45:00,MT,364,266,JACIARA,INGESTÃO DE ÁLCOOL PELO CONDUTOR,COLISÃO COM OBJETO,...,UOP01-DEL02-MT,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,2,8,1,False
4,762973,2026-04-05,DOMINGO,07:50:00,RS,448,22,PORTO ALEGRE,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO COM OBJETO,...,UOP01-DEL01-RS,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,4,5,1,False


## Estrutura dos dados

Antes de criar a tabela Delta é importante verificar os tipos das colunas.

In [3]:
silver.info()

<class 'pandas.DataFrame'>
RangeIndex: 23475 entries, 0 to 23474
Data columns (total 39 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   id                      23475 non-null  int64         
 1   data_inversa            23475 non-null  datetime64[us]
 2   dia_semana              23475 non-null  str           
 3   horario                 23475 non-null  str           
 4   uf                      23475 non-null  str           
 5   br                      23475 non-null  int64         
 6   km                      23475 non-null  str           
 7   municipio               23475 non-null  str           
 8   causa_acidente          23475 non-null  str           
 9   tipo_acidente           23475 non-null  str           
 10  classificacao_acidente  23474 non-null  str           
 11  fase_dia                23475 non-null  str           
 12  sentido_via             23475 non-null  str           
 1

## Criação da pasta Delta

Caso exista uma tabela Delta anterior, ela será removida para evitar conflitos de versões.

In [4]:
CAMINHO_DELTA = "../datalake/delta/acidentes"

if os.path.exists(CAMINHO_DELTA):
    shutil.rmtree(CAMINHO_DELTA)

print("Tabela Delta preparada.")

Tabela Delta preparada.


## Criação da tabela Delta Lake

Os dados da camada Silver serão gravados em formato Delta.

Essa será a versão 0 da tabela.

In [5]:
write_deltalake(
    CAMINHO_DELTA,
    silver,
    mode="overwrite"
)

print("Versão 0 criada.")

Versão 0 criada.


## Conferindo a versão criada

In [6]:
dt = DeltaTable(CAMINHO_DELTA)

print("Versão atual:")

dt.version()

Versão atual:


0

## Atualizando alguns registros

Serão alterados alguns registros para demonstrar o funcionamento do Time Travel.

In [7]:
silver2 = silver.copy()

silver2.loc[
    silver2.index[:100],
    "classificacao_acidente"
] = "GRAVE"

silver2.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,uop,_source_file,_batch_id,_ingested_at,nova_coluna,ano,mes,dia,total_vitimas,acidente_grave
0,753147,2026-02-18,QUARTA-FEIRA,13:40:00,RS,158,"565,9",SANTANA DO LIVRAMENTO,DESRESPEITAR A PREFERÊNCIA NO CRUZAMENTO,COLISÃO TRANSVERSAL,...,UOP01-DEL11-RS,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,2,18,2,False
1,758570,2026-03-15,DOMINGO,12:30:00,MG,40,701,BARBACENA,AUSÊNCIA DE REAÇÃO DO CONDUTOR,COLISÃO LATERAL MESMO SENTIDO,...,UOP02-DEL05-MG,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,3,15,1,False
2,747911,2026-01-24,SÁBADO,16:50:00,MG,381,"438,3",SANTA LUZIA,PISTA ESBURACADA,SAÍDA DE LEITO CARROÇÁVEL,...,UOP01-DEL01-MG,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,1,24,2,False
3,751057,2026-02-08,DOMINGO,20:45:00,MT,364,266,JACIARA,INGESTÃO DE ÁLCOOL PELO CONDUTOR,COLISÃO COM OBJETO,...,UOP01-DEL02-MT,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,2,8,1,False
4,762973,2026-04-05,DOMINGO,07:50:00,RS,448,22,PORTO ALEGRE,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO COM OBJETO,...,UOP01-DEL01-RS,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,4,5,1,False


## Criando uma nova versão

Após as alterações será criada uma nova versão da tabela Delta.

In [8]:
write_deltalake(
    CAMINHO_DELTA,
    silver2,
    mode="overwrite"
)

print("Nova versão criada.")

Nova versão criada.


## Histórico da tabela Delta


In [9]:
dt = DeltaTable(CAMINHO_DELTA)

pd.DataFrame(
    dt.history()
)

,timestamp,operation,operationParameters,engineInfo,operationMetrics,clientVersion,version
0,1782445525610,WRITE,{'mode': 'Overwrite'},delta-rs:py-1.6.0,"{'execution_time_ms': 62, 'num_added_files': 1...",delta-rs.py-1.6.0,1
1,1782445525409,WRITE,{'mode': 'Overwrite'},delta-rs:py-1.6.0,"{'execution_time_ms': 66, 'num_added_files': 1...",delta-rs.py-1.6.0,0


## Time Travel

Agora será realizada a leitura da versão 0 da tabela.

In [10]:
versao0 = DeltaTable(
    CAMINHO_DELTA,
    version=0
).to_pandas()

versao0.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,uop,_source_file,_batch_id,_ingested_at,nova_coluna,ano,mes,dia,total_vitimas,acidente_grave
0,753147,2026-02-18,QUARTA-FEIRA,13:40:00,RS,158,"565,9",SANTANA DO LIVRAMENTO,DESRESPEITAR A PREFERÊNCIA NO CRUZAMENTO,COLISÃO TRANSVERSAL,...,UOP01-DEL11-RS,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,2,18,2,False
1,758570,2026-03-15,DOMINGO,12:30:00,MG,40,701,BARBACENA,AUSÊNCIA DE REAÇÃO DO CONDUTOR,COLISÃO LATERAL MESMO SENTIDO,...,UOP02-DEL05-MG,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,3,15,1,False
2,747911,2026-01-24,SÁBADO,16:50:00,MG,381,"438,3",SANTA LUZIA,PISTA ESBURACADA,SAÍDA DE LEITO CARROÇÁVEL,...,UOP01-DEL01-MG,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,1,24,2,False
3,751057,2026-02-08,DOMINGO,20:45:00,MT,364,266,JACIARA,INGESTÃO DE ÁLCOOL PELO CONDUTOR,COLISÃO COM OBJETO,...,UOP01-DEL02-MT,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,2,8,1,False
4,762973,2026-04-05,DOMINGO,07:50:00,RS,448,22,PORTO ALEGRE,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO COM OBJETO,...,UOP01-DEL01-RS,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,4,5,1,False


## Leitura da versão mais recente

In [11]:
versao1 = DeltaTable(
    CAMINHO_DELTA,
    version=1
).to_pandas()

versao1.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,uop,_source_file,_batch_id,_ingested_at,nova_coluna,ano,mes,dia,total_vitimas,acidente_grave
0,753147,2026-02-18,QUARTA-FEIRA,13:40:00,RS,158,"565,9",SANTANA DO LIVRAMENTO,DESRESPEITAR A PREFERÊNCIA NO CRUZAMENTO,COLISÃO TRANSVERSAL,...,UOP01-DEL11-RS,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,2,18,2,False
1,758570,2026-03-15,DOMINGO,12:30:00,MG,40,701,BARBACENA,AUSÊNCIA DE REAÇÃO DO CONDUTOR,COLISÃO LATERAL MESMO SENTIDO,...,UOP02-DEL05-MG,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,3,15,1,False
2,747911,2026-01-24,SÁBADO,16:50:00,MG,381,"438,3",SANTA LUZIA,PISTA ESBURACADA,SAÍDA DE LEITO CARROÇÁVEL,...,UOP01-DEL01-MG,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,1,24,2,False
3,751057,2026-02-08,DOMINGO,20:45:00,MT,364,266,JACIARA,INGESTÃO DE ÁLCOOL PELO CONDUTOR,COLISÃO COM OBJETO,...,UOP01-DEL02-MT,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,2,8,1,False
4,762973,2026-04-05,DOMINGO,07:50:00,RS,448,22,PORTO ALEGRE,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO COM OBJETO,...,UOP01-DEL01-RS,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,4,5,1,False


## Comparação entre as versões

Nesta etapa será possível verificar que apenas alguns registros foram alterados.

In [12]:
print("Versão 0")

display(
    versao0["classificacao_acidente"]
    .value_counts()
)

print()

print("Versão 1")

display(
    versao1["classificacao_acidente"]
    .value_counts()
)

Versão 0


classificacao_acidente
COM VÍTIMAS FERIDAS    18268
SEM VÍTIMAS             3557
COM VÍTIMAS FATAIS      1599
GRAVE                     50
Name: count, dtype: int64


Versão 1


classificacao_acidente
COM VÍTIMAS FERIDAS    18183
SEM VÍTIMAS             3546
COM VÍTIMAS FATAIS      1595
GRAVE                    150
Name: count, dtype: int64

## Schema Evolution

Será adicionada uma nova coluna na tabela Delta sem perder o histórico das versões.

In [13]:
silver3 = silver2.copy()

silver3["origem"] = "DATATRAN_PRF"

write_deltalake(
    CAMINHO_DELTA,
    silver3,
    mode="overwrite",
    schema_mode="merge"
)

print("Nova coluna adicionada.")

Nova coluna adicionada.


## Conferindo o novo Schema

In [14]:
dt = DeltaTable(CAMINHO_DELTA)

print(dt.schema().json())

{'type': 'struct', 'fields': [{'name': 'id', 'type': 'long', 'nullable': True, 'metadata': {}}, {'name': 'data_inversa', 'type': 'timestamp_ntz', 'nullable': True, 'metadata': {}}, {'name': 'dia_semana', 'type': 'string', 'nullable': True, 'metadata': {}}, {'name': 'horario', 'type': 'string', 'nullable': True, 'metadata': {}}, {'name': 'uf', 'type': 'string', 'nullable': True, 'metadata': {}}, {'name': 'br', 'type': 'long', 'nullable': True, 'metadata': {}}, {'name': 'km', 'type': 'string', 'nullable': True, 'metadata': {}}, {'name': 'municipio', 'type': 'string', 'nullable': True, 'metadata': {}}, {'name': 'causa_acidente', 'type': 'string', 'nullable': True, 'metadata': {}}, {'name': 'tipo_acidente', 'type': 'string', 'nullable': True, 'metadata': {}}, {'name': 'classificacao_acidente', 'type': 'string', 'nullable': True, 'metadata': {}}, {'name': 'fase_dia', 'type': 'string', 'nullable': True, 'metadata': {}}, {'name': 'sentido_via', 'type': 'string', 'nullable': True, 'metadata': 

## Visualização da nova coluna

In [15]:
dt.to_pandas().head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,_source_file,_batch_id,_ingested_at,nova_coluna,ano,mes,dia,total_vitimas,acidente_grave,origem
0,753147,2026-02-18,QUARTA-FEIRA,13:40:00,RS,158,"565,9",SANTANA DO LIVRAMENTO,DESRESPEITAR A PREFERÊNCIA NO CRUZAMENTO,COLISÃO TRANSVERSAL,...,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,2,18,2,False,DATATRAN_PRF
1,758570,2026-03-15,DOMINGO,12:30:00,MG,40,701,BARBACENA,AUSÊNCIA DE REAÇÃO DO CONDUTOR,COLISÃO LATERAL MESMO SENTIDO,...,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,3,15,1,False,DATATRAN_PRF
2,747911,2026-01-24,SÁBADO,16:50:00,MG,381,"438,3",SANTA LUZIA,PISTA ESBURACADA,SAÍDA DE LEITO CARROÇÁVEL,...,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,1,24,2,False,DATATRAN_PRF
3,751057,2026-02-08,DOMINGO,20:45:00,MT,364,266,JACIARA,INGESTÃO DE ÁLCOOL PELO CONDUTOR,COLISÃO COM OBJETO,...,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,2,8,1,False,DATATRAN_PRF
4,762973,2026-04-05,DOMINGO,07:50:00,RS,448,22,PORTO ALEGRE,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO COM OBJETO,...,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,4,5,1,False,DATATRAN_PRF


# Conclusão

Neste notebook foi demonstrado:

- Criação da tabela Delta Lake;
- Controle de versões;
- Time Travel;
- Schema Evolution.

Essas funcionalidades permitem rastrear alterações e manter o histórico dos dados de forma segura.